# Statistical Model Comparison (Phase 7)

This notebook answers two questions with formal statistical tests, using the per-fold results produced by Pipelines 1 and 2 (`results/pipeline1/fold_results.csv`, `results/pipeline2/fold_results.csv`):

1. **Friedman test + Nemenyi post-hoc** — are the differences in test RMSE between models (within each dataset) statistically significant?
2. **Wilcoxon signed-rank** — does Multi-output regression have a significant advantage over Single-output regression, per model and dataset (paired over identical seed/fold blocks)?

All tables are saved to `results/statistical_tests/`, and the last cell shows the complete combined results.

Setup: repo imports, pandas display options, load the Pipeline 1 & 2 fold-level results.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if (Path.cwd() / '..' / 'src').resolve().exists() else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

from src.config import get_path
from src.evaluation import friedman_test, nemenyi_posthoc, wilcoxon_pairwise
from src.utils.io import save_table

results_dir = get_path('results_dir')
p1 = pd.read_csv(results_dir / 'pipeline1' / 'fold_results.csv')
p2 = pd.read_csv(results_dir / 'pipeline2' / 'fold_results.csv')
out_dir = results_dir / 'statistical_tests'
print('Pipeline 1 fold rows:', len(p1))
print('Pipeline 2 fold rows:', len(p2))

## Friedman test across models (per dataset)

Blocks = (target, seed, fold); columns = models; values = test RMSE. A significant Friedman result (p < 0.05) means at least one model differs from the rest.

In [ ]:
friedman_rows = []
nemenyi_tables = {}
for ds_name, grp in p1.groupby('dataset'):
    matrix = grp.pivot_table(
        index=['target', 'seed', 'fold'], columns='model', values='test_rmse'
    ).dropna()
    res = friedman_test(matrix)
    friedman_rows.append({'dataset': ds_name, **res})
    nemenyi = nemenyi_posthoc(matrix)
    nemenyi.index = nemenyi.columns = matrix.columns
    nemenyi_tables[ds_name] = nemenyi
    save_table(nemenyi, out_dir / f'nemenyi_{ds_name}.csv', index=True)
    print(f'[friedman] {ds_name}: statistic={res["statistic"]:.4f}, p_value={res["p_value"]:.6f}')

friedman_df = pd.DataFrame(friedman_rows)
save_table(friedman_df, out_dir / 'friedman_across_models.csv')
friedman_df

### Nemenyi post-hoc pairwise p-values — Dataset_0136

In [ ]:
nemenyi_tables['Dataset_0136']

### Nemenyi post-hoc pairwise p-values — Dataset_0172

In [ ]:
nemenyi_tables['Dataset_0172']

### Nemenyi post-hoc pairwise p-values — Dataset_3772

In [ ]:
nemenyi_tables['Dataset_3772']

## Wilcoxon signed-rank — Single-output vs Multi-output

Paired per (dataset, model, seed, fold): compares the mean test RMSE of the Single-output pipeline against the Multi-output pipeline. A significant result (p < 0.05) means the mode matters for that model on that dataset.

In [ ]:
single = (
    p1.groupby(['dataset', 'model', 'seed', 'fold'])['test_rmse'].mean().rename('single')
)
multi = (
    p2.groupby(['dataset', 'model', 'seed', 'fold'])['test_rmse'].mean().rename('multi')
)
paired = pd.concat([single, multi], axis=1).dropna().reset_index()

wilcoxon_rows = []
for (ds_name, model), grp in paired.groupby(['dataset', 'model']):
    res = wilcoxon_pairwise(grp['single'], grp['multi'])
    wilcoxon_rows.append(
        {
            'dataset': ds_name,
            'model': model,
            'mean_rmse_single': grp['single'].mean(),
            'mean_rmse_multi': grp['multi'].mean(),
            **res,
        }
    )

wilcoxon_df = pd.DataFrame(wilcoxon_rows)
save_table(wilcoxon_df, out_dir / 'wilcoxon_single_vs_multi.csv')
wilcoxon_df.sort_values('p_value')

## Final Summary — Complete Statistical Test Results

Friedman + Nemenyi + Wilcoxon tables displayed in full.

In [ ]:
print('=' * 90)
print('1) FRIEDMAN TEST ACROSS MODELS (per dataset)')
print('=' * 90)
display(friedman_df)

for ds_name, table in nemenyi_tables.items():
    print()
    print('=' * 90)
    print(f'2) NEMENYI POST-HOC P-VALUES — {ds_name}')
    print('=' * 90)
    display(table)

print()
print('=' * 90)
print('3) WILCOXON SIGNED-RANK — Single-output vs Multi-output')
print('=' * 90)
display(wilcoxon_df)

print()
print('All statistical test tables saved to:', out_dir.resolve())